# 📘 CodeBook Social Network Analysis
## Notebook 04 — Pages You Might Like

**Author:** Aditi Chaudhary  
**Tech Stack:** Pure Python · JSON

---

## 🎯 What is 'Pages You Might Like'?

This is the page and content recommendation feature you see on  
Facebook, LinkedIn, and YouTube ('Channels you might like').

The key idea:
> **If your friends like a page you haven't seen yet,  
> there's a good chance you'll like it too.**

---

## 🧠 Algorithm: Friend-Based Collaborative Filtering

```
For the target user:
  1. Get the list of pages they already like  → exclude these
  2. Get their friends list
  3. For each friend:
       For each page the friend likes:
           If target doesn't already like it:
               page_score[page] += 1   (one friend vote)
  4. Sort pages by score (most friend-votes first)
  5. Return top N
```

### Why is this called Collaborative Filtering?

We're using the **collective behaviour of similar users** (friends)  
to recommend **items** (pages) to a target user.

This is exactly how **Netflix recommends movies** and **Spotify recommends songs**.  
The only difference is that Netflix uses millions of users — we use friends.

---

## 📂 In This Notebook

1. Load cleaned data
2. Walk through the algorithm manually step by step
3. Run `pages_you_might_like()` for multiple users
4. Analyse which pages are most recommended across the network

In [ ]:
import sys, os, json

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.data_loader    import load_data
from src.recommendation import (
    build_user_map,
    build_page_map,
    pages_you_might_like,
    display_page_recommendations
)

cleaned_path = os.path.join(project_root, 'data', 'cleaned_data.json')
cleaned_data = load_data(cleaned_path)

users    = cleaned_data['users']
pages    = cleaned_data['pages']
user_map = build_user_map(users)
page_map = build_page_map(pages)

print(f'Loaded {len(users)} users and {len(pages)} pages.')

---
## 🔍 Step 1 — Manual Walkthrough for Aarav Sharma (u001)

Let's trace through the algorithm by hand before running the function.

In [ ]:
target = user_map['u001']   # Aarav Sharma

print(f"Target user : {target['name']}")
print(f"Friends     : {target['friends']}")
print(f"Already likes pages: {target['liked_pages']}")

# Show what the liked pages are called
print('\nAarav already likes:')
for pid in target['liked_pages']:
    page = page_map.get(pid, {})
    print(f'  [{pid}] {page.get("name", "Unknown")}')

In [ ]:
# Now look at what friends like
already_liked = set(target['liked_pages'])
page_votes    = {}

print("\nLooking at what Aarav's friends like...\n")

for friend_id in target['friends']:
    friend = user_map.get(friend_id)
    if not friend:
        continue

    print(f"  Friend: {friend['name']} likes pages: {friend['liked_pages']}")

    for pid in friend['liked_pages']:
        if pid in already_liked:
            print(f'    ↳ [{pid}] Aarav already likes this — skipping')
            continue

        if pid not in page_votes:
            page_votes[pid] = {'votes': 0, 'liked_by': []}
        page_votes[pid]['votes']    += 1
        page_votes[pid]['liked_by'].append(friend['name'])
        print(f'    ↳ [{pid}] Vote added! (score now: {page_votes[pid]["votes"]})')

print('\n📊 Final Page Scores for Aarav:')
for pid, data in sorted(page_votes.items(), key=lambda x: -x[1]['votes']):
    pname = page_map.get(pid, {}).get('name', 'Unknown')
    print(f'  [{pid}] {pname:<35} | Votes: {data["votes"]} | By: {data["liked_by"]}')

---
## 🚀 Step 2 — Run `pages_you_might_like()` for Multiple Users

In [ ]:
# Recommendations for Aarav Sharma
display_page_recommendations('u001', user_map, page_map, top_n=3)

In [ ]:
# Recommendations for Priya Mehta
display_page_recommendations('u002', user_map, page_map, top_n=3)

In [ ]:
# Recommendations for Vikram Rao
display_page_recommendations('u008', user_map, page_map, top_n=3)

---
## 📊 Step 3 — Which Pages Are Most Recommended Across the Network?

Let's find which pages get recommended to the most users — these are the  
**most viral pages** on CodeBook.

In [ ]:
# Count how many times each page appears in recommendations across all users
page_recommendation_count = {}

for uid in user_map:
    recs = pages_you_might_like(uid, user_map, page_map, top_n=5)
    for rec in recs:
        pid = rec['page_id']
        page_recommendation_count[pid] = page_recommendation_count.get(pid, 0) + 1

# Sort by count
sorted_pages = sorted(page_recommendation_count.items(),
                      key=lambda x: x[1], reverse=True)

print('\n📈 Most Recommended Pages Across the Entire Network:')
print('=' * 60)
for pid, count in sorted_pages:
    pname    = page_map.get(pid, {}).get('name', 'Unknown')
    category = page_map.get(pid, {}).get('category', '?')
    bar      = '█' * count
    print(f'  [{pid}] {pname:<35} | Recommended to {count} user(s) | {bar}')

---
## 🔑 Step 4 — Key Insights from Page Recommendations

In [ ]:
# Find the most recommended page
if sorted_pages:
    top_pid, top_count = sorted_pages[0]
    top_page = page_map.get(top_pid, {})
    
    print('\n🏆 Insights:')
    print(f'  Most recommended page : {top_page.get("name")}')
    print(f'  Recommended to        : {top_count} users')
    print(f'  Category              : {top_page.get("category")}')
    print(f'  Followers             : {top_page.get("followers", 0):,}')
    print()
    print('  📌 Business Insight:')
    print('     Pages that appear most in recommendations are the best candidates')
    print('     for sponsored/promoted placement — they already have organic reach.')

---
## ✅ Notebook 04 — Summary

| Concept | What We Learned |
|---|---|
| Collaborative Filtering | Use friend behaviour to predict user preferences |
| Friend Votes | More friends liked a page = higher recommendation score |
| Exclusion Logic | Already-liked pages are excluded from recommendations |
| Network Analysis | Found the most 'viral' pages across the whole network |
| Real-world link | Netflix, Spotify, YouTube use this same core idea |

---

## 🏁 Project Complete!

| Notebook | Feature | Status |
|---|---|---|
| 01 | Data Loading | ✅ |
| 02 | Data Cleaning (4 steps) | ✅ |
| 03 | People You May Know | ✅ |
| 04 | Pages You Might Like | ✅ |

**Skills demonstrated:** Pure Python, JSON parsing, set operations,  
social network analysis, mutual friend algorithm, collaborative filtering,  
data cleaning, modular code design.